**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP ENVIRONMENT**

In [ ]:
# PARAMETRI JOB
job_name = "autocast-v8"    # nome da passare poi a notebook moco 
dataset = "Test"            # Test, Standard, Anomalies*
time_debug = False          # time_debug = True solo per debug, = False per training

encoders_train_func = project.new_function(
    name= f'encoders_[Floods_{dataset}]-[{job_name}]',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="pretrain_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "400Gi"}   
    }
]

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")


**TRAINING**

In [ ]:
parametri = {
    "epochs": 3, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # ottiche               
    "mamba": False, 
    "workers": 0,
    "patience": 20,
    "job_name": job_name,
    "dataset": dataset,
    "time_debug": time_debug
}

print(f"PARAMETRI: {parametri}")

# action job = avvia container, esegui script, libera risorse

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_encoders avviato: {run_train_encoders.id}")

In [ ]:
# rieseguibile
run_train_encoders.refresh()    
print(run_train_encoders.status.state)
print(run_train_encoders.status.message)
# print(run_train_encoders.logs())

**PLOTS**

In [ ]:
# salvataggio log
path_s1 = project.get_artifact(f"metrics-s1_{job_name}").download(overwrite=True)
path_s2 = project.get_artifact(f"metrics-s2_{job_name}").download(overwrite=True)

df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

# plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
plt.title(f"Training Encoders - {job_name}")

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title(f'SAR')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title(f'OTTICO')
ax2.set_xlabel('Epochs')
ax2.grid(True)

plt.show()
